In [ ]:
# Cell 1: Install Required Libraries
!pip install geemap earthengine-api geopandas rasterio scikit-learn -q

In [ ]:
# Cell 2: Import Required Libraries
import ee
import geemap
import json
import geopandas as gpd
import rasterio

In [ ]:
# Cell 3: Authenticate Google Earth Engine
ee.Authenticate()

In [ ]:
# Cell 4: Initialize Google Earth Engine
PROJECT = "seagrass-uae-project"

ee.Initialize(project=PROJECT)

print(" Earth Engine initialized successfully!")

In [ ]:
# Cell 5: Load Study Area from Uploaded GeoJSON
import json

study_area_path = "/content/study_area.geojson"

with open(study_area_path) as f:
    geojson = json.load(f)

study_area = ee.Geometry(geojson["features"][0]["geometry"])

print(" Study area loaded successfully!")

In [ ]:
# Cell 6: Display Study Area
Map = geemap.Map()

Map.centerObject(study_area, 10)

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map

In [ ]:
# Cell 7: Load Sentinel-2 Surface Reflectance Collection
start_date = "2024-01-01"
end_date   = "2024-12-31"

sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(study_area)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
)

print("Number of images:", sentinel2.size().getInfo())

In [ ]:
# Cell 8: Create Median Composite Image
composite = sentinel2.median().clip(study_area)

print("Composite image created successfully!")

In [ ]:
# Cell 9: Display Sentinel-2 Composite
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

Map = geemap.Map()

Map.centerObject(study_area, 11)

Map.addLayer(composite, rgb_vis, "Sentinel-2 RGB")

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map

In [ ]:

# Cell 9 optional: Display Sentinel-2 Composite


rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

Map = geemap.Map()

# Use a satellite basemap
Map.add_basemap("SATELLITE")

Map.centerObject(study_area, 11)

Map.addLayer(
    composite,
    rgb_vis,
    "Sentinel-2 RGB"
)

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map

In [ ]:
# Cell 10: Define Sentinel-2 Cloud Mask Function
def mask_s2_clouds(image):
    """
    Masks clouds and cirrus using the QA60 band.
    """
    qa = image.select("QA60")

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    return (
        image.updateMask(mask)
             .divide(10000)
             .copyProperties(image, ["system:time_start"])
    )

In [ ]:
# Cell 11: Apply Cloud Mask and Create Clean Composite
clean_collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(study_area)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .map(mask_s2_clouds)
)

clean_composite = clean_collection.median().clip(study_area)

print(" Cloud-free composite created successfully!")

In [ ]:
# Cell 12: Display Cloud-Free Sentinel-2 Composite
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0.0,
    "max": 0.30
}

Map = geemap.Map()

Map.add_basemap("SATELLITE")

Map.centerObject(study_area, 11)

Map.addLayer(
    clean_composite,
    rgb_vis,
    "Cloud-Free Sentinel-2"
)

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map

In [ ]:
# Cell 13: Calculate NDVI (Normalized Difference Vegetation Index)
ndvi = clean_composite.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI")

print(" NDVI created successfully!")

In [ ]:
# Cell 14: Display NDVI
ndvi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["blue", "white", "green"]
}

# Create the interactive map
Map = geemap.Map()

Map.add_basemap("SATELLITE")

Map.centerObject(study_area, 11)

# Add NDVI layer
Map.addLayer(
    ndvi,
    ndvi_vis,
    "NDVI"
)
# Add study area boundary, only some of them cover the picture
outline = ee.Image().byte().paint(
    featureCollection=study_area,
    color=1,
    width=2
)
# Add study area
Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

# Enable layer control
Map.addLayerControl()

Map

In [ ]:
# Cell 15: Calculate NDWI (Normalized Difference Water Index)
ndwi = clean_composite.normalizedDifference(
    ["B3", "B8"]
).rename("NDWI")

print(" NDWI created successfully!")

In [ ]:
# ============================================================
# Cell 16: Add NDWI Layer
# ============================================================

ndwi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["brown", "white", "blue"]
}

Map.addLayer(
    ndwi,
    ndwi_vis,
    "NDWI"
)

Map

In [ ]:
# Cell 17: Calculate Modified NDWI (MNDWI)
mndwi = clean_composite.normalizedDifference(
    ["B3", "B11"]
).rename("MNDWI")

print(" MNDWI created successfully!")

In [ ]:
# Cell 18: Add MNDWI Layer
# ============================================================

mndwi_vis = {
    "min": -1,
    "max": 1,
    "palette": ["brown", "white", "cyan"]
}

Map.addLayer(
    mndwi,
    mndwi_vis,
    "MNDWI"
)

Map

In [ ]:
# Cell 19: Calculate Bare Soil Index (BSI)
bsi = clean_composite.expression(
    "((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))",
    {
        "SWIR": clean_composite.select("B11"),
        "RED": clean_composite.select("B4"),
        "NIR": clean_composite.select("B8"),
        "BLUE": clean_composite.select("B2")
    }
).rename("BSI")

print(" BSI created successfully!")

In [ ]:
# Cell 20: Display Bare Soil Index (BSI)
bsi_vis = {
    "min": -1,
    "max": 1,
    "palette": [
        "blue",
        "white",
        "brown"
    ]
}

Map = geemap.Map()

Map.centerObject(study_area, 11)

Map.addLayer(
    bsi,
    bsi_vis,
    "BSI"
)

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map

In [ ]:
# Cell 21: Build Feature Stack
feature_stack = clean_composite.addBands([
    ndvi,
    ndwi,
    mndwi,
    bsi
])

print(" Feature stack created successfully!")

In [ ]:
# Cell 22: Check Feature Stack Bands
print(feature_stack.bandNames().getInfo())


In [ ]:
# Cell 23: Display Feature Stack (RGB)
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 0.30
}

Map = geemap.Map()

Map.centerObject(study_area, 11)

Map.addLayer(
    feature_stack,
    rgb_vis,
    "Feature Stack RGB"
)

Map.addLayer(
    study_area,
    {"color": "red"},
    "Study Area"
)

Map.addLayerControl()

Map

In [ ]:
# A7: Acquire Official EAD Seagrass Habitat Reference Data

In [ ]:
# Connect to the Environment Agency – Abu Dhabi (EAD) Habitat Service

# This section retrieves the official seagrass habitat reference data from the Environment Agency – Abu Dhabi (EAD).
 # These reference polygons will be used to generate training samples for supervised Random Forest classification.

In [ ]:
# Cell 24: Import Libraries for EAD Habitat Data
import requests
import geopandas as gpd

In [ ]:
# Cell 25: Define the EAD Marine Habitat Service
ead_url = (
    "https://arcgis.sdi.abudhabi.ae/agspublish/rest/services/"
    "OpenData/ADSDI_OpenData/MapServer/141/query"
)

print(" EAD Marine Habitat service defined.")

In [ ]:
# Cell 26: Download Seagrass Habitat Polygons
params = {
    "where": "HABITATTYPE='Seagrass bed'",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "geojson"
}

response = requests.get(ead_url, params=params)

print(response.status_code)

In [ ]:
# Cell 27: Save Seagrass Habitat GeoJSON
with open("ead_seagrass.geojson", "w", encoding="utf-8") as f:
    f.write(response.text)

print(" GeoJSON saved successfully!")

In [ ]:
# Cell 28: Load Official EAD Seagrass Data
ead_seagrass = gpd.read_file("ead_seagrass.geojson")

print("Total EAD seagrass polygons:", len(ead_seagrass))

ead_seagrass.head()

In [ ]:
# Cell 29: Load Study Area GeoJSON
study_area_gdf = gpd.read_file("study_area.geojson")

print(study_area_gdf.crs)
study_area_gdf.head()

In [ ]:
# Cell 30: Verify Coordinate Reference Systems (CRS)
print("Study Area CRS :", study_area_gdf.crs)
print("EAD Seagrass CRS:", ead_seagrass.crs)

# Reproject only if necessary
if study_area_gdf.crs != ead_seagrass.crs:
    ead_seagrass = ead_seagrass.to_crs(study_area_gdf.crs)
    print(" CRS matched.")
else:
    print(" Both datasets already use the same CRS.")

In [ ]:
# Cell 31: Clip Official EAD Seagrass to Study Area
ead_marawah = gpd.clip(ead_seagrass, study_area_gdf)

print("Official EAD seagrass polygons inside study area:", len(ead_marawah))

In [ ]:
# Cell 32: Plot Clipped Seagrass Polygons
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,10))

study_area_gdf.boundary.plot(
    ax=ax,
    color="red",
    linewidth=2,
    label="Study Area"
)

ead_marawah.plot(
    ax=ax,
    color="green",
    alpha=0.6,
    edgecolor="black",
    label="Official EAD Seagrass"
)

plt.title("Official EAD Seagrass within Marawah Study Area")
plt.legend()

plt.show()

In [ ]:
# Cell 33: Save Clipped EAD Seagrass Reference Data
ead_marawah.to_file(
    "marawah_ead_seagrass_reference.geojson",
    driver="GeoJSON"
)

print(" Saved:", "marawah_ead_seagrass_reference.geojson")

In [ ]:
!ls

In [ ]:
#A8 – Prepare the Random Forest Training Dataset

In [ ]:
# Cell 34: Download Official Non-Seagrass Habitat
import requests

params = {
    "where": "HABITATTYPE = 'Unconsolidated bottom'",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "geojson"
}

response = requests.get(ead_url, params=params)

print("HTTP Status:", response.status_code)

with open("ead_non_seagrass.geojson", "w") as f:
    f.write(response.text)

print(" Official non-seagrass habitat downloaded.")

In [ ]:
# Cell 35: Load Official Non-Seagrass Habitat
non_seagrass = gpd.read_file("ead_non_seagrass.geojson")

print("Total official non-seagrass polygons:", len(non_seagrass))

non_seagrass.head()

In [ ]:
# Cell 36: Clip Non-Seagrass Habitat to the Study Area
# Clip official unconsolidated bottom polygons
non_seagrass_marawah = gpd.clip(non_seagrass, study_area_gdf)

print("Official non-seagrass polygons inside study area:",
      len(non_seagrass_marawah))

non_seagrass_marawah.head()

In [ ]:
# Cell 37: Assign Training Class Labels
# Seagrass = 1
ead_marawah["class"] = 1

# Non-seagrass = 0
non_seagrass_marawah["class"] = 0

print("Seagrass class assigned.")
print("Non-seagrass class assigned.")

# Check the new column
print("\nSeagrass:")
print(ead_marawah[["HABITATTYPE", "class"]].head())

print("\nNon-Seagrass:")
print(non_seagrass_marawah[["HABITATTYPE", "class"]].head())

In [ ]:
# Cell 38: Save Prepared Training Classes
ead_marawah.to_file(
    "marawah_seagrass_training.geojson",
    driver="GeoJSON"
)

non_seagrass_marawah.to_file(
    "marawah_non_seagrass_training.geojson",
    driver="GeoJSON"
)

print(" Training datasets saved.")
print("Seagrass polygons:", len(ead_marawah))
print("Non-seagrass polygons:", len(non_seagrass_marawah))

In [ ]:
# Module A9: Random Forest Classification (Original Dataset)


In [ ]:
# Cell 40: Merge Original Training Dataset
import pandas as pd

# Merge seagrass and non-seagrass polygons
training_gdf = pd.concat(
    [ead_marawah, non_seagrass_marawah],
    ignore_index=True
)

print("===================================")
print("Original Training Dataset")
print("===================================")
print(f"Total polygons: {len(training_gdf)}")
print()

print(training_gdf["class"].value_counts())

In [ ]:
# ============================================================
# Cell 41: Convert Training Data to Earth Engine
# ============================================================

import json
import ee

# Keep only the required columns
ead_clean = ead_marawah[["class", "geometry"]].copy()
non_clean = non_seagrass_marawah[["class", "geometry"]].copy()

# Convert to GeoJSON dictionaries
seagrass_geojson = json.loads(ead_clean.to_json())
non_seagrass_geojson = json.loads(non_clean.to_json())

# Convert to Earth Engine FeatureCollections
ee_seagrass = ee.FeatureCollection(seagrass_geojson)
ee_non_seagrass = ee.FeatureCollection(non_seagrass_geojson)

# Merge classes
ee_training = ee_seagrass.merge(ee_non_seagrass)

print("Earth Engine training dataset created.")
print("Training features:", ee_training.size().getInfo())

In [ ]:
# Cell 42: Create Predictor Image
# Predictor bands for Random Forest
predictor_bands = [
    "B2", "B3", "B4",
    "B5", "B6", "B7",
    "B8", "B8A",
    "B11", "B12",
    "NDVI",
    "NDWI",
    "MNDWI",
    "BSI"
]

# Keep only predictor bands
predictor_image = feature_stack.select(predictor_bands)

print("Predictor image created successfully.")
print("Predictor bands:")
print(predictor_image.bandNames().getInfo())

In [ ]:
# ============================================================
# Cell 43: Generate Random Training Points
# ============================================================

# Number of points for each class
SEAGRASS_POINTS = 500
NON_SEAGRASS_POINTS = 500

# Generate random points inside seagrass polygons
seagrass_points = ee.FeatureCollection.randomPoints(
    region=ee_seagrass.geometry(),
    points=SEAGRASS_POINTS,
    seed=42
).map(lambda f: f.set("class", 1))

# Generate random points inside non-seagrass polygons
non_seagrass_points = ee.FeatureCollection.randomPoints(
    region=ee_non_seagrass.geometry(),
    points=NON_SEAGRASS_POINTS,
    seed=42
).map(lambda f: f.set("class", 0))

# Merge both classes
training_points = seagrass_points.merge(non_seagrass_points)

print("Training points created.")
print(training_points.size().getInfo())

In [ ]:
# ============================================================
# Cell 44: Sample Predictor Values
# ============================================================

training_samples = predictor_image.sampleRegions(
    collection=training_points,
    properties=["class"],
    scale=10,
    geometries=False
)

print("Training samples extracted successfully.")
print("Number of samples:", training_samples.size().getInfo())

# Display the first sample
print(training_samples.first().getInfo())

In [ ]:
# ============================================================
# Cell 45: Split Training and Testing Data
# ============================================================

# Add a random number to each sample
samples = training_samples.randomColumn("random", seed=42)

# 80% for training
training_set = samples.filter(ee.Filter.lt("random", 0.8))

# 20% for testing
testing_set = samples.filter(ee.Filter.gte("random", 0.8))

print("Training samples:", training_set.size().getInfo())
print("Testing samples :", testing_set.size().getInfo())

In [ ]:
# ============================================================
# Cell 46: Train Random Forest Classifier
# ============================================================

# Train the Random Forest classifier
rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=training_set,
    classProperty="class",
    inputProperties=predictor_bands
)

print("Random Forest classifier trained successfully.")

In [ ]:
# ============================================================
# Cell 47: Classify Predictor Image
# ============================================================

# Apply the trained Random Forest classifier
classified_image = predictor_image.classify(rf_classifier)

print("Image classified successfully.")

In [ ]:
# ============================================================
# Cell 48: Classify Testing Samples
# ============================================================

# Predict the class of the testing samples
classified_test = testing_set.classify(rf_classifier)

print("Testing samples classified successfully.")
print("Testing samples:", classified_test.size().getInfo())

In [ ]:
# ============================================================
# Cell 49: Display Classification Map
# ============================================================

# Visualization parameters
classification_vis = {
    "min": 0,
    "max": 1,
    "palette": [
        "brown",      # 0 = Non-seagrass
        "limegreen"   # 1 = Seagrass
    ]
}

# Create interactive map
import geemap
Map = geemap.Map()

Map.add_basemap("SATELLITE")

# Center on study area
Map.centerObject(study_area, 11)

# Add classified image
Map.addLayer(
    classified_image,
    classification_vis,
    "Random Forest Classification"
)

# Add study area boundary
outline = ee.Image().byte().paint(
    featureCollection=study_area,
    color=1,
    width=2
)

Map.addLayer(
    outline,
    {"palette": ["red"]},
    "Study Area"
)

Map

In [ ]:
# ============================================================
# Cell 50: Accuracy Assessment
# ============================================================

# Compute confusion matrix
confusion_matrix = classified_test.errorMatrix(
    "class",
    "classification"
)

print("Confusion Matrix")
print(confusion_matrix.getInfo())

print("\nOverall Accuracy:")
print(confusion_matrix.accuracy().getInfo())

print("\nKappa Coefficient:")
print(confusion_matrix.kappa().getInfo())

print("\nProducer's Accuracy:")
print(confusion_matrix.producersAccuracy().getInfo())

print("\nUser's Accuracy:")
print(confusion_matrix.consumersAccuracy().getInfo())

In [ ]:
# Original Confusion Matrix
import pandas as pd

original_cm = pd.DataFrame(
    confusion_matrix.getInfo(),
    index=["Actual Non-Seagrass", "Actual Seagrass"],
    columns=["Predicted Non-Seagrass", "Predicted Seagrass"]
)

print("Original Confusion Matrix\n")
display(original_cm)

In [ ]:
# Cell 51: Random Forest Classification Summary
print("=" * 65)
print("      RANDOM FOREST CLASSIFICATION SUMMARY")
print("=" * 65)

print("\nTraining Dataset")
print("-" * 30)
print(f"Training Samples : {training_set.size().getInfo()}")
print(f"Testing Samples  : {testing_set.size().getInfo()}")

print("\nModel Parameters")
print("-" * 30)
print("Algorithm        : Random Forest")
print("Decision Trees   : 100")
print("Predictor Bands  : 14")
print("Random Seed      : 42")

print("\nClassification Accuracy")
print("-" * 30)
print(f"Overall Accuracy : {confusion_matrix.accuracy().getInfo()*100:.2f}%")
print(f"Kappa Coefficient: {confusion_matrix.kappa().getInfo():.3f}")

print("\nProducer's Accuracy")
print("-" * 30)
pa = confusion_matrix.producersAccuracy().getInfo()
print(f"Non-Seagrass     : {pa[0][0]*100:.2f}%")
print(f"Seagrass         : {pa[1][0]*100:.2f}%")

print("\nUser's Accuracy")
print("-" * 30)
ua = confusion_matrix.consumersAccuracy().getInfo()
print(f"Non-Seagrass     : {ua[0][0]*100:.2f}%")
print(f"Seagrass         : {ua[0][1]*100:.2f}%")

print("\nStatus")
print("-" * 30)
print("✓ Random Forest model trained successfully")
print("✓ Classification completed")
print("✓ Accuracy assessment completed")
print("✓ Baseline model ready for comparison")

print("=" * 65)

In [ ]:
#A10: Balanced Random Forest Classification

In [ ]:
# Cell 52: Create Balanced Training Dataset

import pandas as pd

# Number of seagrass polygons
n_seagrass = len(ead_marawah)

# Randomly select the same number of non-seagrass polygons
balanced_non_seagrass = non_seagrass_marawah.sample(
    n=n_seagrass,
    random_state=42
)

# Merge into one balanced dataset
balanced_training_gdf = pd.concat(
    [ead_marawah, balanced_non_seagrass],
    ignore_index=True
)

print("Balanced training dataset created.\n")
print(f"Seagrass polygons     : {len(ead_marawah)}")
print(f"Non-seagrass polygons : {len(balanced_non_seagrass)}")
print(f"Total polygons        : {len(balanced_training_gdf)}")

print("\n✓ Ready for Earth Engine.")

In [ ]:
# Cell 53: Convert Balanced Dataset to Earth Engine
import json

# Separate classes
balanced_seagrass = balanced_training_gdf[
    balanced_training_gdf["class"] == 1
][["class", "geometry"]].copy()

balanced_non_seagrass = balanced_training_gdf[
    balanced_training_gdf["class"] == 0
][["class", "geometry"]].copy()

# Convert to GeoJSON
balanced_seagrass_json = json.loads(balanced_seagrass.to_json())
balanced_non_seagrass_json = json.loads(balanced_non_seagrass.to_json())

# Convert to Earth Engine FeatureCollections
ee_balanced_seagrass = ee.FeatureCollection(balanced_seagrass_json)
ee_balanced_non_seagrass = ee.FeatureCollection(balanced_non_seagrass_json)

# Merge
ee_balanced_training = ee_balanced_seagrass.merge(
    ee_balanced_non_seagrass
)

print("Earth Engine FeatureCollection created.\n")
print(f"Training features : {ee_balanced_training.size().getInfo()}")

print("\n✓ Conversion successful.")

In [ ]:
# Cell 54: Generate Balanced Random Training Points
SEAGRASS_POINTS = 500
NON_SEAGRASS_POINTS = 500

balanced_seagrass_points = ee.FeatureCollection.randomPoints(
    region=ee_balanced_seagrass.geometry(),
    points=SEAGRASS_POINTS,
    seed=42
).map(lambda f: f.set("class", 1))

balanced_non_seagrass_points = ee.FeatureCollection.randomPoints(
    region=ee_balanced_non_seagrass.geometry(),
    points=NON_SEAGRASS_POINTS,
    seed=42
).map(lambda f: f.set("class", 0))

balanced_training_points = balanced_seagrass_points.merge(
    balanced_non_seagrass_points
)

print("Random training points generated.\n")
print(f"Seagrass points     : {SEAGRASS_POINTS}")
print(f"Non-seagrass points : {NON_SEAGRASS_POINTS}")
print(f"Total points        : {balanced_training_points.size().getInfo()}")

print("\n✓ Ready for sample extraction.")

In [ ]:
# Cell 55: Extract Predictor Values
balanced_training_samples = predictor_image.sampleRegions(
    collection=balanced_training_points,
    properties=["class"],
    scale=10,
    geometries=False
)

print("Training samples extracted.\n")
print(f"Samples : {balanced_training_samples.size().getInfo()}")

print("\n✓ Ready for model training.")

In [ ]:
# Cell 56: Split Training and Testing
balanced_samples = balanced_training_samples.randomColumn(
    "random",
    seed=42
)

balanced_training_set = balanced_samples.filter(
    ee.Filter.lt("random", 0.8)
)

balanced_testing_set = balanced_samples.filter(
    ee.Filter.gte("random", 0.8)
)

print("Training / Testing split completed.\n")

print(f"Training : {balanced_training_set.size().getInfo()}")
print(f"Testing  : {balanced_testing_set.size().getInfo()}")

print("\n Ready to train the model.")

In [ ]:
# Cell 57: Train Balanced Random Forest
balanced_rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=balanced_training_set,
    classProperty="class",
    inputProperties=predictor_bands
)

print("Balanced Random Forest model trained.\n")
print("Trees : 100")

print("\n Model ready for classification.")

In [ ]:
# Cell 58: Classify Image
balanced_classified_image = predictor_image.classify(
    balanced_rf_classifier
)

print("Image classified successfully.")

print("\n✓ Classification completed.")

In [ ]:
# Cell 59: Display Classification Map
classification_vis = {
    "min": 0,
    "max": 1,
    "palette": ["brown", "limegreen"]
}

Map = geemap.Map()

Map.add_basemap("SATELLITE")

Map.centerObject(study_area, 11)

Map.addLayer(
    balanced_classified_image,
    classification_vis,
    "Balanced RF Classification"
)

outline = ee.Image().byte().paint(
    featureCollection=study_area,
    color=1,
    width=2
)

Map.addLayer(
    outline,
    {"palette": ["red"]},
    "Study Area"
)

print("Classification map displayed.\n")
print("Green : Seagrass")
print("Brown : Non-seagrass")

Map

In [ ]:
# Cell 60: Classify Testing Samples
balanced_classified_test = balanced_testing_set.classify(
    balanced_rf_classifier
)

print("Testing samples classified.\n")

print(
    f"Samples : {balanced_classified_test.size().getInfo()}"
)

print("\n✓ Ready for accuracy assessment.")

In [ ]:
# Cell 61: Accuracy Assessment
balanced_confusion_matrix = balanced_classified_test.errorMatrix(
    "class",
    "classification"
)

# Accuracy metrics
oa = balanced_confusion_matrix.accuracy().getInfo()
kappa = balanced_confusion_matrix.kappa().getInfo()
pa = balanced_confusion_matrix.producersAccuracy().getInfo()
ua = balanced_confusion_matrix.consumersAccuracy().getInfo()

print("Accuracy assessment completed.\n")

print(f"Overall Accuracy : {oa*100:.2f}%")
print(f"Kappa            : {kappa:.3f}")

print("\nProducer's Accuracy")
print(f"  Non-Seagrass : {pa[0][0]*100:.2f}%")
print(f"  Seagrass     : {pa[1][0]*100:.2f}%")

print("\nUser's Accuracy")
print(f"  Non-Seagrass : {ua[0][0]*100:.2f}%")
print(f"  Seagrass     : {ua[0][1]*100:.2f}%")

print("\n✓ Model evaluation completed.")

In [ ]:
# Balanced Confusion Matrix

balanced_cm = pd.DataFrame(
    balanced_confusion_matrix.getInfo(),
    index=["Actual Non-Seagrass", "Actual Seagrass"],
    columns=["Predicted Non-Seagrass", "Predicted Seagrass"]
)

print("Balanced Confusion Matrix\n")
display(balanced_cm)

In [ ]:
# Cell 62: Compare Original and Balanced Models
import pandas as pd

original_pa = confusion_matrix.producersAccuracy().getInfo()
original_ua = confusion_matrix.consumersAccuracy().getInfo()

balanced_pa = balanced_confusion_matrix.producersAccuracy().getInfo()
balanced_ua = balanced_confusion_matrix.consumersAccuracy().getInfo()

comparison = pd.DataFrame({
    "Metric":[
        "Overall Accuracy (%)",
        "Kappa",
        "Producer Accuracy (Non-Seagrass)",
        "Producer Accuracy (Seagrass)",
        "User Accuracy (Non-Seagrass)",
        "User Accuracy (Seagrass)"
    ],

    "Original":[
        round(confusion_matrix.accuracy().getInfo()*100,2),
        round(confusion_matrix.kappa().getInfo(),3),
        round(original_pa[0][0]*100,2),
        round(original_pa[1][0]*100,2),
        round(original_ua[0][0]*100,2),
        round(original_ua[0][1]*100,2)
    ],

    "Balanced":[
        round(balanced_confusion_matrix.accuracy().getInfo()*100,2),
        round(balanced_confusion_matrix.kappa().getInfo(),3),
        round(balanced_pa[0][0]*100,2),
        round(balanced_pa[1][0]*100,2),
        round(balanced_ua[0][0]*100,2),
        round(balanced_ua[0][1]*100,2)
    ]
})

comparison

In [ ]:
# Cell 63: Feature Importance
importance = ee.Dictionary(
    balanced_rf_classifier.explain().get("importance")
)

importance_df = (
    pd.DataFrame.from_dict(
        importance.getInfo(),
        orient="index",
        columns=["Importance"]
    )
    .sort_values("Importance", ascending=False)
)

importance_df

In [ ]:
# Cell 64: Feature Importance Plot
import matplotlib.pyplot as plt

importance_df.plot(
    kind="bar",
    figsize=(9,5),
    legend=False
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Predictor")
plt.ylabel("Importance")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# A11 Export Final Results

In [ ]:
#Cell 65 – Export Original Classification
task = ee.batch.Export.image.toDrive(
    image=classified_image,
    description="RF_Original_Classification",
    folder="dugong_uae",
    fileNamePrefix="rf_original",
    region=study_area,
    scale=10,
    maxPixels=1e13
)

task.start()

print("Original classification export started.")

In [ ]:
# Cell 66 – Export Balanced Classification
task = ee.batch.Export.image.toDrive(
    image=balanced_classified_image,
    description="RF_Balanced_Classification",
    folder="dugong_uae",
    fileNamePrefix="rf_balanced",
    region=study_area,
    scale=10,
    maxPixels=1e13
)

task.start()

print("Balanced classification export started.")

In [ ]:
# Cell 67: Save Results
comparison.to_csv(
    "rf_model_comparison.csv",
    index=False
)

importance_df.to_csv(
    "feature_importance.csv"
)

print("Results saved successfully.")